# zagg paired read — coincident ATL03 × GEDI

Two sensors, one grid. `serc_tdigest_strata.zarr` holds ATL03 signal-stratum
photon digests at **o19** cells; `serc_gedi_flux.zarr` holds GEDI L1B
waveform flux digests at **o18** cells. Both were built on the same o9 SERC
shards, so pairing is pure morton arithmetic: an o12 block covers a 128×128
o19 tensor on the ATL03 side and a 64×64 o18 tensor on the GEDI side, and one
GEDI cell is exactly the 2×2 ATL03 cells beneath it.

Three linked views, each with one set of controls driving both panes:
elevation slices, the 3-D block, and cell-level waveforms.

In [1]:
# %pip install "moczarr>=0.4.0" "zagg[catalog,viz]" ipympl

import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm
from moczarr.convention import morton_decimal
from moczarr.hhdc import read_tensors
from moczarr.open import open_leaf
from mortie import generate_morton_children

from zagg.catalog.shardmap import ShardMap
from zagg.config import default_config
from zagg.readers._layout import rowcol_to_rank
from zagg.readers.tdigest_tensor import cell_index, read_cell
from zagg.stats.tdigest import cdf_from_tdigest

# prefer the 'nasa' profile when it exists (laptop); ambient creds otherwise (hub role)
import botocore.session as _bs

if "nasa" in (_bs.Session().full_config.get("profiles") or {}):
    os.environ.setdefault("AWS_PROFILE", "nasa")
OUT = Path("outputs")
STORE = "s3://sliderule-public/zagg-demo"
ATL03 = f"{STORE}/serc_tdigest_strata.zarr"
GEDI = f"{STORE}/serc_gedi_flux.zarr"

timings = {}


class stage:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        timings[self.name] = round(time.perf_counter() - self.t0, 2)
        print(f"[{self.name}] {timings[self.name]:.1f}s")

## Coincidence is free

The two shardmaps carry the **same o9 shard keys** — same AOI, same grid, same
sharding — so a paired read is two `open_leaf` calls on one key. Blocks join
on their o12 morton word; cells join by index arithmetic (`(r, c)` at o18 ↔
`(2r+δr, 2c+δc)` at o19). Ranking below is by *joint* occupancy — GEDI is a
coverage instrument, ICESat-2 a track instrument, so the interesting cells
are where a ground track crosses sampled footprints.

In [2]:
acfg = default_config("atl03_tdigest_healpix_hive")
gcfg = default_config("gedi01b_waveform_healpix_hive")
AFIELD = f"{acfg.output['grid']['child_order']}/h_tdigest_signal"
GFIELD = f"{gcfg.output['grid']['child_order']}/rx_flux"

sm_a = ShardMap.from_json(str(OUT / "shardmap_atl03_serc_o9.json"))
sm_g = ShardMap.from_json(str(OUT / "shardmap_gedi_serc_o9.json"))
common = sorted(set(map(int, sm_a.shard_keys)) & set(map(int, sm_g.shard_keys)))
shard = max(common, key=lambda k: len(sm_g.granules[list(map(int, sm_g.shard_keys)).index(k)]))
print(f"{len(common)} shared o9 shards; reading {shard}")

astore = open_leaf(ATL03, shard)
gstore = open_leaf(GEDI, shard)
with stage("paired tensors: one shard, both sensors"):
    ablocks = {
        b[3]: b
        for b in read_tensors(
            astore, AFIELD, n_bins=256, resolution=1.0, block_order=12, fit="degrade_resolution"
        )
    }
    gblocks = {
        b[3]: b
        for b in read_tensors(
            gstore, GFIELD, n_bins=256, resolution=1.0, block_order=12, fit="degrade_resolution"
        )
    }

pairs = []  # (word, joint mask at o18, atl03 colsums folded 2x2, gedi colsums)
for w, (gt, gm, (goff, gg), _) in gblocks.items():
    if w not in ablocks:
        continue
    at = ablocks[w][0]
    A2 = at.sum(axis=2).reshape(64, 2, 64, 2).sum(axis=(1, 3))  # o19 -> o18 footprint
    G2 = gt.sum(axis=2)
    joint = (A2 > 0) & (G2 > 0)
    if joint.any():
        pairs.append((w, joint, A2, G2))
pairs.sort(key=lambda p: -int(p[1].sum()))
pd.DataFrame(
    [
        {
            "block": morton_decimal(w),
            "joint cells (o18)": int(j.sum()),
            "atl03 photons in joint": int(A2[j].sum()),
            "gedi pe in joint": int(G2[j].sum()),
        }
        for w, j, A2, G2 in pairs[:8]
    ]
)

4 shared o9 shards; reading 5347391238804865033
[paired tensors: one shard, both sensors] 11.0s


,block,joint cells (o18),atl03 photons in joint,gedi pe in joint
0,4331422411132,136,10174,1151326
1,4331422411111,109,6886,841698
2,4331422411113,103,6280,1023844
3,4331422411143,90,6128,545831
4,4331422411423,84,3548,187847
5,4331422411114,78,4061,759727
6,4331422411112,77,1608,730076
7,4331422411311,74,5945,708929


## Paired elevation slices

One control row, two panes. The slider walks ATL03's elevation bins; the GEDI
pane follows to its **nearest bin by absolute elevation** (each block's two
tensors bin from their own data-driven offsets). ATL03 unobserved cells and
empty bins are transparent; ditto GEDI.

In [3]:
from ipywidgets import Checkbox, Dropdown, HBox, IntSlider, VBox, interactive_output

woptions = [
    (f"{morton_decimal(w)}  ({int(j.sum())} joint cells)", i) for i, (w, j, _, _) in enumerate(pairs)
]
_AVMAX = float(max(ablocks[w][0].max() for w, _, _, _ in pairs))
_GVMAX = float(max(gblocks[w][0].max() for w, _, _, _ in pairs))


def paired_slice(pair=0, z=32):
    w, joint, A2, G2 = pairs[pair]
    at, am, (aoff, ag), _ = ablocks[w]
    gt, gm, (goff, gg), _ = gblocks[w]
    elev = aoff + (z + 0.5) * ag
    gz = int(np.clip(round((elev - goff) / gg - 0.5), 0, 255))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8))
    sl = at[:, :, z].astype(float)
    sl[(am != 2) | (sl == 0)] = np.nan
    ax1.imshow(sl, origin="lower", cmap="magma", norm=LogNorm(vmin=1, vmax=_AVMAX))
    ax1.set_title(
        f"ATL03 signal — bin {z:03d}: {aoff + z * ag:+.1f} … {aoff + (z + 1) * ag:+.1f} m\n"
        f"{int(np.nansum(sl)):,} photons",
        fontsize=9,
    )
    gl = gt[:, :, gz].astype(float)
    gl[(gm == 0) | (gl == 0)] = np.nan
    ax2.imshow(gl, origin="lower", cmap="viridis", norm=LogNorm(vmin=1, vmax=_GVMAX))
    ax2.set_title(
        f"GEDI flux — bin {gz:03d}: {goff + gz * gg:+.1f} … {goff + (gz + 1) * gg:+.1f} m\n"
        f"{int(np.nansum(gl)):,} photoelectrons",
        fontsize=9,
    )
    for ax, n in ((ax1, 128), (ax2, 64)):
        ax.set_facecolor("#e8e8e8")
        ax.set_xticks([0, n // 2, n - 1])
        ax.set_yticks([0, n // 2, n - 1])
    fig.suptitle(f"block {morton_decimal(w)} — {elev:+.1f} m", fontsize=11)
    plt.tight_layout()
    plt.show()


_pair_dd = Dropdown(options=woptions, value=0, description="block")
_z_sl = IntSlider(min=0, max=255, value=128, description="elev bin")
_out = interactive_output(paired_slice, {"pair": _pair_dd, "z": _z_sl})
display(VBox([HBox([_pair_dd, _z_sl]), _out]))

## The block in 3-D, both sensors — exact by default

Default view reads the digests directly (`read_ragged`): z is the stored
float32 centroid elevation, xy is decoded from the §9 located sibling where
the field carries one (ATL03 — point-exact) and the o18 cell center where it
doesn't (GEDI flux ≈ footprint scale). **binned** switches both axes to the
fixed numpy tensors — binned locations *and* elevations, the rasterized view
the slice explorer uses. The panes' cameras are linked: drag either and both
rotate together. **color by time** decodes the §8.3 temporal sibling's toc
words so acquisition seams render as color bands (the 0.47 GEDI store
carries the channel; an ATL03 store built with the strata temporal wiring in
`02_write` lights up the left pane too). **color by elevation** shares one
height scale; **z extent** pins both panes to one sensor's vertical window.

In [4]:
%matplotlib widget
from moczarr.convention import point_to_area29
from moczarr.ragged import open_ragged, read_ragged
from mortie import rank_to_xy, toc2time

# Default = EXACT: digests read directly, z the stored float32 centroid
# elevation, xy decoded from the located sibling's word where the field
# carries one (ATL03, point-exact) and the cell word where it doesn't (GEDI
# flux ~ footprint scale) -- both in the BLOCK'S OWN LATTICE FRAME, the same
# square the binned tensors plot in. "binned" switches to the fixed tensors
# (binned locations AND elevations). Everything is PREFETCHED in one
# whole-shard sweep per store, so block switching is instant.
_CAP = 20000
_cent_cache = {}
# ~side of an o12 HEALPix cell (equal-area; square-equivalent), for extent axes
_SIDE12 = float(np.sqrt(4 * np.pi / (12 * 4**12)) * 6_371_000)


def _grid_xy(words, block_order=12):
    """Fractional (row, col) of each word UNDER its o12 block, in meters.

    Vectorized spec-section-1 bit decode (validated bit-exact against the
    per-digit path decode): the body holds 2-bit path ranks at fixed slots
    (containment = truncation), and the 28-47 suffix band carries the
    level-28/29 digits arithmetically. A word keeps its own order's
    precision -- an o29 point resolves to ~1 cm, a merged centroid's coarser
    DCA word renders at that cell's center.
    """
    w = np.asarray(words, dtype=np.uint64)
    s = (w & np.uint64(63)).astype(np.int64)
    o = np.where(s <= 27, s, np.where((s - 28) % 5 == 0, 28, 29)).astype(np.int64)
    x = np.zeros(len(w)); y = np.zeros(len(w))
    for L in range(block_order + 1, 30):
        m = np.flatnonzero(o >= L)
        if not len(m):
            continue
        if L <= 27:
            rank = ((w[m] >> np.uint64(6 + 2 * (27 - L))) & np.uint64(3)).astype(np.int64)
        elif L == 28:
            rank = (s[m] - 28) // 5
        else:
            rank = (s[m] - 28) % 5 - 1
        cx, cy = rank_to_xy(rank, 1)
        scale = 2.0 ** -(L - block_order)
        x[m] += np.asarray(cy, dtype=float) * scale  # row -> x (module orientation)
        y[m] += np.asarray(cx, dtype=float) * scale
    half = 0.5 * (2.0 ** -(o - block_order).astype(float))
    return (x + half) * _SIDE12, (y + half) * _SIDE12


def _prefetch_centroids():
    if _cent_cache:
        return
    joint = {int(w) for w, _, _, _ in pairs}
    with stage("prefetch exact centroids: whole shard, both sensors"):
        for key, store, field in (("atl03", astore, AFIELD), ("gedi", gstore, GFIELD)):
            arr, _ = open_ragged(store, field)
            attrs = dict(arr.attrs)
            locname = (attrs.get("ragged") or {}).get("locations")
            tname = attrs.get("times")
            zs, wts, cellw, locw, cell_seq = [], [], [], [], []
            for row in read_ragged(store, field, locations=bool(locname)):
                word, vals = row[0], row[1]
                cell_seq.append(word)
                zs.append(vals[:, 0]); wts.append(vals[:, 1])
                cellw.append(np.full(len(vals), word, dtype=np.uint64))
                if locname:
                    locw.append(np.asarray(row[2], dtype=np.uint64))
            z = np.concatenate(zs); wt = np.concatenate(wts)
            cells = np.concatenate(cellw)
            # bucket by o12 ancestor: containment is truncation (spec
            # section 1 / section 4) -- zero the path slots below level 12
            # and stamp the order-12 suffix (validated exact)
            _sh = np.uint64(6 + 2 * (27 - 12))
            blocks = ((cells >> _sh) << _sh) | np.uint64(12)
            if locname:
                gx, gy = _grid_xy(point_to_area29(np.concatenate(locw)))
                xy_note = "exact xy"
            else:
                gx, gy = _grid_xy(cells)
                xy_note = "cell-center xy"
            t_days = None
            if tname:
                tfield = field.rsplit("/", 1)[0] + "/" + tname
                tmap = {int(r[0]): np.asarray(r[1], dtype=np.uint64).ravel()
                        for r in read_ragged(store, tfield)}
                # one vector per CELL, in the payload sweep's cell order (the
                # sibling is row-aligned per cell)
                tw = np.concatenate([tmap[int(cw)] for cw in cell_seq])
                start_ns, _end = toc2time(tw)
                # toc scale = ns since 1850-01-01 (GPS-aligned; mortie
                # time2toc contract) -- re-zero to the 2018 mission epoch
                _ns2018 = float((np.datetime64("2018-01-01") - np.datetime64("1850-01-01")) // np.timedelta64(1, "ns"))
                t_days = (np.asarray(start_ns, dtype="float64") - _ns2018) / 86.4e12
            for w in joint:
                m = np.flatnonzero(blocks == np.uint64(w))
                if len(m) > _CAP:
                    m = m[np.random.default_rng(w & 0xFFFF).choice(len(m), _CAP, replace=False)]
                _cent_cache.setdefault(w, {})[key] = {
                    "z": z[m], "wt": wt[m], "x": gx[m], "y": gy[m],
                    "xy_note": xy_note,
                    "t_days": None if t_days is None else t_days[m],
                }


def _binned_pts(t, off, g, n, cap=_CAP):
    xs, ys, zs = np.nonzero(t)
    counts = t[xs, ys, zs].astype("float64")
    if len(xs) > cap:
        keep = np.random.default_rng(0).choice(len(xs), cap, replace=False)
        xs, ys, zs, counts = xs[keep], ys[keep], zs[keep], counts[keep]
    return xs / n, ys / n, off + zs * g, counts


def paired_3d(pair=0, by_elev=False, by_time=False, binned=False, zmode="auto"):
    w = pairs[pair][0]
    if binned:
        at, _, (aoff, ag), _ = ablocks[w]
        gt, _, (goff, gg), _ = gblocks[w]
        panes = [dict(zip(("x", "y", "z", "wt"), _binned_pts(at, aoff, ag, 128)),
                      t_days=None, xy_note=f"binned ({ag:g} m z)", label="ATL03 signal photons"),
                 dict(zip(("x", "y", "z", "wt"), _binned_pts(gt, goff, gg, 64)),
                      t_days=None, xy_note=f"binned ({gg:g} m z)", label="GEDI photoelectrons")]
        for p_ in panes:
            p_["x"] = p_["x"] * _SIDE12
            p_["y"] = p_["y"] * _SIDE12
    else:
        _prefetch_centroids()
        data = _cent_cache[int(w)]
        panes = [dict(x=d["x"], y=d["y"], z=d["z"], wt=d["wt"], t_days=d["t_days"],
                      xy_note=d["xy_note"], label=lbl)
                 for d, lbl in ((data["atl03"], "ATL03 signal photons"),
                                (data["gedi"], "GEDI flux centroids"))]
    zlim = {"auto": None,
            "atl03": (panes[0]["z"].min(), panes[0]["z"].max()),
            "gedi": (panes[1]["z"].min(), panes[1]["z"].max())}[zmode]
    if by_elev and not by_time:
        shared_elev = plt.Normalize(min(p_["z"].min() for p_ in panes), max(p_["z"].max() for p_ in panes))
    t_avail = [p_["t_days"] for p_ in panes if p_["t_days"] is not None]
    if by_time and t_avail:
        shared_time = plt.Normalize(min(t.min() for t in t_avail), max(t.max() for t in t_avail))
    ext = float(max(max(p_["x"].max(), p_["y"].max()) for p_ in panes))
    _km = ext >= 2000
    _tickv = [0.0, ext / 2, ext]
    _tickl = [f"{v / 1000:.2f} km" if _km else f"{v:.0f} m" for v in _tickv]
    fig = plt.figure(figsize=(11, 5.2))
    axes = []
    for k, (pane, cmap) in enumerate(zip(panes, ("viridis", "plasma"))):
        ax = fig.add_subplot(1, 2, k + 1, projection="3d")
        axes.append(ax)
        note = f" — {pane['xy_note']}"
        alpha = np.clip(pane["wt"] / max(np.percentile(pane["wt"], 98), 1e-9), 0.08, 1.0)
        if by_time:
            if pane["t_days"] is not None:
                pts = ax.scatter(pane["x"], pane["y"], pane["z"], c=pane["t_days"], s=1.5,
                                 cmap="turbo", norm=shared_time, alpha=alpha)
                cb = fig.colorbar(pts, shrink=0.55, pad=0.10)
                cb.set_label("acquisition (days since 2018-01-01)")
            else:
                ax.scatter(pane["x"], pane["y"], pane["z"], color="#9498a0", s=1.5, alpha=0.15)
                note += " — no temporal channel" + (" in binned tensors" if binned else " in this store")
        elif by_elev:
            pts = ax.scatter(pane["x"], pane["y"], pane["z"], c=pane["z"], s=1.5,
                             cmap="viridis", norm=shared_elev, alpha=alpha)
            fig.colorbar(pts, shrink=0.55, pad=0.10, label="elevation (m)")
        else:
            pts = ax.scatter(pane["x"], pane["y"], pane["z"], c=pane["wt"], s=1.5,
                             cmap=cmap, norm=LogNorm(), alpha=alpha)
            fig.colorbar(pts, shrink=0.55, pad=0.10,
                         label="photons" if k == 0 else "photoelectrons")
        if zlim is not None:
            ax.set_zlim(*zlim)
        ax.set_zlabel("elevation (m)")
        ax.set_title(pane["label"] + note, fontsize=9)
        ax.set_xticks(_tickv, _tickl, fontsize=7)
        ax.set_yticks(_tickv, _tickl, fontsize=7)
        ax.set_xlabel("east", fontsize=8, labelpad=-2)
        ax.set_ylabel("north", fontsize=8, labelpad=-2)
    # linked rotation: only while DRAGGING (button held), and only when the
    # angles actually moved -- a bare hover must not trigger redraws
    def _sync(event):
        if event.button is None or event.inaxes not in axes:
            return
        src = event.inaxes
        for other in axes:
            if other is not src and (other.elev != src.elev or other.azim != src.azim):
                other.view_init(elev=src.elev, azim=src.azim)
                fig.canvas.draw_idle()
    fig.canvas.mpl_connect("motion_notify_event", _sync)
    fig.suptitle(f"block {morton_decimal(w)}" + ("" if binned else " — exact centroids"), fontsize=11)
    plt.show()


_pair3d_dd = Dropdown(options=woptions, value=0, description="block")
_zmode_dd = Dropdown(
    options=[("independent z", "auto"), ("pin z to ATL03", "atl03"), ("pin z to GEDI", "gedi")],
    value="auto",
    description="z extent",
)
_elev_cb = Checkbox(value=False, description="color by elevation (shared)")
_time_cb = Checkbox(value=False, description="color by time (shared)")
_bin_cb = Checkbox(value=False, description="binned (fixed tensors)")
_out3d = interactive_output(
    paired_3d,
    {"pair": _pair3d_dd, "by_elev": _elev_cb, "by_time": _time_cb, "binned": _bin_cb, "zmode": _zmode_dd},
)
display(VBox([HBox([_pair3d_dd, _zmode_dd]), HBox([_elev_cb, _time_cb, _bin_cb]), _out3d]))

## Coincident waveforms

The cell-level join: one GEDI o18 cell against the 2×2 ATL03 o19 cells under
it, both reconstructed from their stored digests as Gaussian mixtures on a
shared elevation axis. The slider ranks joint cells by the *weaker* member
(`min(photons, pe)`), so early picks have real data on both sides. GEDI's
chunk grid puts one o12 block in one chunk; ATL03's chunks sit at o13.

In [5]:
%matplotlib inline
from ipywidgets import FloatText


def _mixture(digest, z, sigma):
    mu, wt = digest[:, 0], digest[:, 1]
    pdf = (wt[None, :] * np.exp(-0.5 * ((z[:, None] - mu[None, :]) / sigma) ** 2)).sum(axis=1)
    return pdf / max(wt.sum(), 1e-9) / (sigma * np.sqrt(2 * np.pi))


def paired_waveform(pair=0, nth=0, binw=1.0):
    w, joint, A2, G2 = pairs[pair]
    _, _, (aoff, ag), _ = ablocks[w]
    _, _, (goff, gg), _ = gblocks[w]
    rank = np.minimum(A2, G2) * joint
    order = np.argsort(rank.ravel())[::-1]
    r, c = np.unravel_index(int(order[min(nth, int(joint.sum()) - 1)]), rank.shape)

    gdigest = read_cell(gstore, GFIELD, cell_index(gstore, GFIELD, int(w), int(r), int(c)))
    kids = []
    for dr in (0, 1):
        for dc in (0, 1):
            rr, cc = 2 * r + dr, 2 * c + dc
            cid = int(generate_morton_children(int(w), 13)[rowcol_to_rank(rr // 64, cc // 64, depth=1)])
            try:
                kids.append(read_cell(astore, AFIELD, cell_index(astore, AFIELD, cid, rr % 64, cc % 64)))
            except Exception:
                pass
    adigest = np.concatenate([k for k in kids if len(k)]) if kids else np.empty((0, 2))

    lo = min(gdigest[:, 0].min(), adigest[:, 0].min()) - 5
    hi = max(gdigest[:, 0].max(), adigest[:, 0].max()) + 5
    z = np.linspace(lo, hi, 700)
    amu, awt = adigest[:, 0], adigest[:, 1]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
    # bottom x-axis: the two digest-mixture densities (probability / m)
    ax1.plot(_mixture(gdigest, z, gg), z, color="#7b3294", lw=2, label=f"GEDI flux ({gdigest[:, 1].sum():.0f} pe)")
    ax1.plot(_mixture(adigest, z, ag), z, color="#008837", lw=2, label=f"ATL03 signal ({awt.sum():.0f} ph)")
    ax1.set_xlabel("normalized density")
    ax1.set_ylabel("elevation (m)")
    # top x-axis: RAW ATL03 photons -- binned bars at the chosen width, or the
    # individual photons as a column of dots when the width is 0 (cells under
    # the store's delta budget are loss-free, so centroids ~ photons there;
    # merged centroids carry their weight -- dot size shows it)
    ax1t = ax1.twiny()
    if binw and binw > 0:
        edges = np.arange(lo, hi + binw, binw)
        counts, _ = np.histogram(amu, bins=edges, weights=awt)
        ax1t.barh(edges[:-1] + binw / 2, counts, height=binw * 0.9,
                  color="#008837", alpha=0.25, zorder=0)
        ax1t.set_xlabel(f"ATL03 photons / {binw:g} m bin", fontsize=9)
    else:
        ax1t.scatter(np.zeros(len(amu)), amu, s=np.clip(awt * 8, 8, 40),
                     color="#008837", alpha=0.45, marker="o", zorder=0)
        ax1t.set_xlim(-0.05, 1.0)
        ax1t.set_xlabel("ATL03 photons (unbinned)", fontsize=9)
    ax1.set_zorder(ax1t.get_zorder() + 1)
    ax1.patch.set_visible(False)
    ax1.set_title(
        f"cell ({r},{c}) @o18 — GEDI {len(gdigest)} centroids vs ATL03 {len(adigest)} (2×2 @o19)",
        fontsize=9,
    )
    ax1.legend(fontsize=8)
    # cdf_from_tdigest returns CUMULATIVE WEIGHT (pe for GEDI, photon counts
    # for ATL03) -- normalize each by its total so both live in probability
    # space and share the axis honestly.
    ax2.plot(cdf_from_tdigest(gdigest, z) / max(gdigest[:, 1].sum(), 1e-9), z, color="#7b3294", lw=2)
    ax2.plot(cdf_from_tdigest(adigest, z) / max(adigest[:, 1].sum(), 1e-9), z, color="#008837", lw=2)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("CDF (probability)")
    ax2.set_title("cumulative", fontsize=10)
    for ax in (ax1, ax2):
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(alpha=0.25, lw=0.5)
    fig.suptitle(f"block {morton_decimal(w)}", fontsize=11)
    plt.tight_layout()
    plt.show()


_pairw_dd = Dropdown(options=woptions, value=0, description="block")
_nth_sl = IntSlider(min=0, max=40, value=0, description="nth joint")
_binw_ft = FloatText(value=1.0, step=0.5, description="bin (m)")
_outw = interactive_output(paired_waveform, {"pair": _pairw_dd, "nth": _nth_sl, "binw": _binw_ft})
display(VBox([HBox([_pairw_dd, _nth_sl, _binw_ft]), _outw]))

In [ ]:
(OUT / "timings_paired.json").write_text(json.dumps(timings, indent=2))
timings